<a href="https://colab.research.google.com/github/novalrnv/machine_learning/blob/main/JS03/JS03-TugasLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

df = pd.read_csv('/content/wbc.csv')

##1. Pisahkan antara variabel

In [3]:
df = df.drop(columns=['id', 'Unnamed: 32'], errors='ignore')

X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

##2. Proses Encoding

In [4]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

##3. Standarisasi pada semua kolom

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

##4. Proses seleksi fitur

In [6]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=10)

X_train_selected = selector.fit_transform(X_train_scaled, y_train)

X_test_selected = selector.transform(X_test_scaled)

##5. Pengujian dengan Logistic Regression

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(max_iter=1000)

model.fit(X_train_selected, y_train)

y_pred = model.predict(X_test_selected)
akurasi_manual = accuracy_score(y_test, y_pred)

print(f"Akurasi (Manual Step 3-5): {akurasi_manual * 100:.2f}%")

Akurasi (Manual Step 3-5): 97.37%


##6. Menggunakan pipeline

In [8]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif, k=10)),
    ('classifier', LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)

y_pred_pipe = pipeline.predict(X_test)
akurasi_pipeline = accuracy_score(y_test, y_pred_pipe)

print(f"Akurasi (Dengan Pipeline): {akurasi_pipeline * 100:.2f}%")

Akurasi (Dengan Pipeline): 97.37%


##7. Analisa

In [9]:
from sklearn.model_selection import GridSearchCV

total_features = X_train.shape[1]
param_grid = {'selector__k': list(range(1, total_features + 1))}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')

grid_search.fit(X_train, y_train)

best_k = grid_search.best_params_['selector__k']
best_model = grid_search.best_estimator_

selected_mask = best_model.named_steps['selector'].get_support()
selected_features = X.columns[selected_mask]

print("=== HASIL ANALISA LANGKAH 7 ===")
print(f"Jumlah fitur terbaik yang dapat digunakan adalah: {best_k} fitur.")
print(f"Akurasi model dengan {best_k} fitur: {grid_search.best_score_ * 100:.2f}%\n")

print("Fitur-fitur tersebut adalah:")
for i, feature in enumerate(selected_features, 1):
    print(f"{i}. {feature}")

=== HASIL ANALISA LANGKAH 7 ===
Jumlah fitur terbaik yang dapat digunakan adalah: 17 fitur.
Akurasi model dengan 17 fitur: 97.58%

Fitur-fitur tersebut adalah:
1. radius_mean
2. perimeter_mean
3. area_mean
4. compactness_mean
5. concavity_mean
6. concave points_mean
7. radius_se
8. perimeter_se
9. area_se
10. radius_worst
11. texture_worst
12. perimeter_worst
13. area_worst
14. compactness_worst
15. concavity_worst
16. concave points_worst
17. symmetry_worst


###Penjelasan
Berdasarkan hasil analisa:
1. **Berapa jumlah fitur terbaik:** Jumlah fitur terbaik yang dapat digunakan adalah sebanyak 17 Fitur. Fitur-fitur tersebut terpilih karena memiliki korelasi statistik terkuat terhadap kolom diagnosis.

2. **Apa saja fitur tersebut:** Berdasarkan hasil analisa menggunakan `GridSearchCV` dan `SelectKBest`, berikut adalah fitur-fitur terbaik yang memberikan akurasi maksimal pada model Logistic Regression:

| No | Nama Fitur Terbaik |
| :---: | :--- |
| **1** | `radius_mean` |
| **2** | `perimeter_mean` |
| **3** | `area_mean` |
| **4** | `compactness_mean` |
| **5** | `concavity_mean` |
| **6** | `concave points_mean` |
| **7** | `radius_se` |
| **8** | `perimeter_se` |
| **9** | `area_se` |
| **10** | `radiust_worst`|
| **11** | `texture_worst`|
| **12** | `parameter_worst`|
| **13** | `area_worst`|
| **14** | `compactness_worst`|
| **15** | `concavity_worst`|
| **16** | `concave points_worst`|
| **17** | `symmetry_worst`|